In [37]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold

metadata_path = "data/annotations/metadata.csv"

df = pd.read_csv(metadata_path)

In [27]:
def plate_label(row: pd.Series) -> str:
    parts = [row["food_label_1"]]
    if pd.notna(row.get("food_label_2")):
        parts.append(row["food_label_2"])
    if pd.notna(row.get("food_label_3")):
        parts.append(row["food_label_3"])
    return ", ".join(parts)

In [28]:
df["plate"] = df.apply(plate_label, axis=1)
df["portion_key"] = df["plate"] + " | " + df["images"].apply(
    lambda p: Path(p).parent.name
)



In [30]:
train_df, holdout_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df["portion_key"]
)

In [31]:
holdout_df

,food_label_1,weight_1,food_label_2,weight_2,food_label_3,weight_3,images,masks,plate,portion_key
7130,Prosciutto Cotto,80.0,NaN,NaN,NaN,NaN,data/raw/Prosciutto Cotto/80g/ProsciuttoCotto_...,NaN,Prosciutto Cotto,Prosciutto Cotto | 80g
7499,Kiwi,120.0,NaN,NaN,NaN,NaN,data/raw/Kiwi/120g/Kiwi_porzione120gr_Immagine...,NaN,Kiwi,Kiwi | 120g
1896,Mela,120.0,NaN,NaN,NaN,NaN,data/raw/Mela/120g/Mela_porzione120gr_Immagine...,NaN,Mela,Mela | 120g
4798,Pecorino Grattuggiato,60.0,NaN,NaN,NaN,NaN,data/raw/Pecorino Grattuggiato/60g/PecorinoGra...,NaN,Pecorino Grattuggiato,Pecorino Grattuggiato | 60g
654,Frittata,175.0,NaN,NaN,NaN,NaN,data/raw/Frittata/175g/Frittata_175gr_Immagine...,NaN,Frittata,Frittata | 175g
...,...,...,...,...,...,...,...,...,...,...
7447,Ravioli,30.0,NaN,NaN,NaN,NaN,data/raw/Ravioli/30g/Ravioli_porzione30gr_Imma...,NaN,Ravioli,Ravioli | 30g
5600,steak,90.0,mixed salad,95.0,NaN,NaN,"data/raw/steak, mixed salad/90g, 95g/20251124_...",NaN,"steak, mixed salad","steak, mixed salad | 90g, 95g"
6297,salmon,160.0,leek mashed potatoes,253.0,NaN,NaN,"data/raw/salmon, leek mashed potatoes/160g, 25...",NaN,"salmon, leek mashed potatoes","salmon, leek mashed potatoes | 160g, 253g"
6544,chicken,54.0,rice,80.0,currysauce,76.0,"data/raw/chicken, rice, currysauce/54g, 80g, 7...",NaN,"chicken, rice, currysauce","chicken, rice, currysauce | 54g, 80g, 76g"


In [32]:
val_df, test_df = train_test_split(
    holdout_df,
    test_size=0.5,
    random_state=42,
    stratify=holdout_df["portion_key"]
)

In [33]:
val_df = val_df.assign(split="holdout_val")
test_df = test_df.assign(split="holdout_test")

In [35]:
train_df = train_df.copy()
train_df["split"] = None
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df["portion_key"])):
    train_df.iloc[val_idx, train_df.columns.get_loc("split")] = f"fold_{fold}"


In [36]:
final_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

In [38]:
final_df = final_df[["images", "split"]]

In [39]:
final_df.to_csv("data/annotations/final_split.csv", index=False)